# Aquaculture Model Analysis

This notebook demonstrates how to analyze a trained model from the aquaculture competition framework using actual competition data.

In [ ]:
import numpy as np
import pandas as pd
import os
import random
from pathlib import Path
import sys
import re
import joblib
import yaml
import matplotlib.pyplot as plt
import optuna
import optuna.visualization as vis

# Add the parent directory to the system path to import local modules
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

# Import our custom modules
from aquaculture.feature_engineering import AquacultureFeatureEngineer
from aquaculture.config import AquacultureConfig
from src.inference import load_inference_pipeline
from src.plotting import (
    plot_feature_importance, plot_roc_curve, plot_precision_recall_curve,
    plot_confusion_matrix, plot_calibration_curve
)
from src.metrics import calculate_metrics, competition_score, calculate_roc_curve, calculate_precision_recall_curve
from sklearn.metrics import confusion_matrix
from sklearn.calibration import calibration_curve


In [ ]:
# Add the parent directory to the system path to import local modules
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

# For reproducibility
np.random.seed(42)
random.seed(42)

# Set up paths
DATA_DIR = Path('../data')
EXPERIMENTS_DIR = Path('../experiments')

# Try to find the most recent experiment directory
if EXPERIMENTS_DIR.exists():
    experiment_dirs = [d for d in EXPERIMENTS_DIR.iterdir() if d.is_dir()]
    if experiment_dirs:
        # Sort by modification time (newest first)
        experiment_dirs.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        latest_experiment = experiment_dirs[0]
        print(f"Found experiment: {latest_experiment.name}")
    else:
        print("No experiment directories found!")
        sys.exit(1)
else:
    print("Experiments directory not found!")
    sys.exit(1)

# Load the inference pipeline
print("Loading inference pipeline...")
try:
    pipeline = load_inference_pipeline(str(latest_experiment))
    print("✓ Inference pipeline loaded successfully")
    print(f"Model type: {type(pipeline.model).__name__}")
    if pipeline.feature_names:
        print(f"Number of features: {len(pipeline.feature_names)}")
except Exception as e:
    print(f"Error loading model: {e}")
    print("Please check that the experiment directory exists and contains a trained model")
    sys.exit(1)


## 2. Load Features and Generate Predictions

Load the engineered features and true labels, then generate predictions using the loaded model pipeline.

In [ ]:
# Load training data from CSV file
print("Loading training data...")
train_df = pd.read_csv(DATA_DIR / 'Train.csv')
print(f"Training data shape: {train_df.shape}")
print(f"Training data columns: {list(train_df.columns)}")

# Prepare data for training
print("Preparing data for training...")
# The target column is 'label' in the training data
# Feature columns are all columns except ID and label
feature_cols = [col for col in train_df.columns if col not in ['ID', 'label']]
X_flat = train_df[feature_cols].values  # raw feature matrix (flattened)
# Reshape to 3D as expected by the feature engineer: (n_samples, 12, 12)
n_samples = X_flat.shape[0]
X = X_flat.reshape(n_samples, 12, 12)
# Get target variable - binary classification: 0 (no pond) or 1 (pond)
print("Extracting target variable from 'label' column...")
y = train_df['label'].values
print(f"Target variable shape: {y.shape}")
print(f"Target distribution: {np.bincount(y.astype(int)) if len(y) > 0 else 'empty'}")


## 3. Re‑create engineered features from raw data (training mode)


In [ ]:
# Load experiment config to get feature engineering settings
config_path = latest_experiment / "config.yaml"
with open(config_path, 'r') as f:
    # Use FullLoader to allow construction of Python tuples (e.g., window_length_probs)
    config_dict = yaml.load(f, Loader=yaml.FullLoader)
# The TrainingConfig stores feature_engineering_config as a dict
feat_cfg_dict = config_dict.get('feature_engineering_config', {})
# If it's empty, we can also try to load via TrainingConfig (but it may not have the attr)
feature_engineering_config = AquacultureConfig(**feat_cfg_dict) if feat_cfg_dict else AquacultureConfig()
print(f"Loaded feature engineering config: {feature_engineering_config}")


In [ ]:
# Try to get the already‑fitted feature engineer from the trainer
trainer_path = latest_experiment / "trainer.pkl"
feature_engineer = None
if trainer_path.exists():
    try:
        with open(trainer_path, 'rb') as f:
            trainer_obj = pickle.load(f)
        # The trainer may have a feature_engineer attribute
        if hasattr(trainer_obj, 'feature_engineer') and trainer_obj.feature_engineer is not None:
            feature_engineer = trainer_obj.feature_engineer
            print("Retrieved fitted feature engineer from trainer.pkl")
        else:
            print("Trainer loaded but no feature_engineer attribute found.")
    except Exception as e:
        print(f"Failed to load trainer.pkl: {e}")
else:
    print("trainer.pkl not found.")

# If we don't have a fitted engineer, create a fresh one and fit it
if feature_engineer is None:
    feature_engineer = AquacultureFeatureEngineer(
        simulate_mask=feature_engineering_config.simulate_mask,
        random_state=feature_engineering_config.random_state,
        window_length_probs=feature_engineering_config.window_length_probs,
        start_month_distribution=feature_engineering_config.start_month_distribution,
        s2_monthly_dropout=feature_engineering_config.s2_monthly_dropout,
        include_optical=feature_engineering_config.include_optical,
        include_sar=feature_engineering_config.include_sar,
        include_cross_sensor_features=feature_engineering_config.include_cross_sensor_features,
        include_temporal_statistics=feature_engineering_config.include_temporal_statistics,
        include_metadata=feature_engineering_config.include_metadata
    )
    print("Created new AquacultureFeatureEngineer instance.")
    # Fit on raw data (just to set internal shapes/feature names)
    feature_engineer.fit(X)
    print("Fitted feature engineer on raw data.")


In [ ]:
# Transform raw data using the feature engineer with training=True
# This applies stochastic window selection and S2‑band dropout.
X_features = feature_engineer.transform(X, training=True)
X_features = X_features.values  # ensure numpy array
print(f"Reconstructed feature matrix shape: {X_features.shape}")
# Optional: show first few feature names
if hasattr(feature_engineer, 'feature_names_out_'):
    print(f"First 5 feature names: {list(feature_engineer.feature_names_out_)[:5]}")
elif hasattr(feature_engineer, 'get_feature_names_out'):
    try:
        names = feature_engineer.get_feature_names_out()
        print(f"First 5 feature names: {list(names)[:5]}")
    except Exception:
        pass


## 4. Continue with model evaluation and visualizations (using reconstructed features)


In [ ]:
# Load labels (we already have X from earlier cells)
labels_df = pd.read_csv(DATA_DIR / "Train.csv")
y = labels_df["label"].values
print(f"Labels shape: {y.shape}")

# Make predictions using the reconstructed features
print("Generating predictions...")
# Use the model directly to avoid double feature transformation
predictions = pipeline.model.predict(X_features)
probabilities = pipeline.model.predict_proba(X_features)[:, 1]
# Note: predict_proba returns shape (n_samples, 2); we take column 1 for positive class


## 5. Evaluate Model Performance


In [ ]:
# Calculate metrics for single target
print(f"\n=== Target Evaluation ===")
target_pred = predictions
target_prob = probabilities
target_true = y

# Calculate various metrics
metrics = calculate_metrics(target_true, target_prob)

# Print key metrics
print(f"Accuracy:  {metrics['accuracy']:.4f}")
print(f"Precision: {metrics['precision']:.4f}")
print(f"Recall:    {metrics['recall']:.4f}")
print(f"F1-Score:  {metrics['f1']:.4f}")
print(f"ROC AUC:   {metrics['roc_auc']:.4f}")
print(f"PR AUC:    {metrics['pr_auc']:.4f}")

# Calculate competition score (for single target, this is just the standard competition score)
competition_score_value = competition_score(target_true, target_prob)
print(f"\nCompetition Score: {competition_score_value:.4f}")

## 6. Generate Visualizations


In [ ]:
# Generate plots for single target
print(f"\nGenerating plots for target...")
target_pred = predictions
target_prob = probabilities
target_true = y

# Create a directory for plots
PLOTS_DIR = EXPERIMENTS_DIR / latest_experiment.name / 'plots'
PLOTS_DIR.mkdir(exist_ok=True)

# ROC Curve
fpr, tpr, _ = calculate_roc_curve(target_true, target_prob)
roc_auc = metrics['roc_auc']
plot_roc_curve(fpr, tpr, roc_auc,
               title=f'ROC Curve',
               save_path=PLOTS_DIR / f'roc_curve.png')

# Precision-Recall Curve
precision, recall, _ = calculate_precision_recall_curve(target_true, target_prob)
pr_auc = metrics['pr_auc']
plot_precision_recall_curve(precision, recall, pr_auc,
                            title=f'Precision-Recall Curve',
                            save_path=PLOTS_DIR / f'pr_curve.png')

# Confusion Matrix
cm = confusion_matrix(target_true, target_pred)
plot_confusion_matrix(cm,
                      title=f'Confusion Matrix',
                      save_path=PLOTS_DIR / f'confusion_matrix.png')

# Calibration Curve
prob_true, prob_pred = calibration_curve(target_true, target_prob, n_bins=10)
plot_calibration_curve(prob_true, prob_pred,
                       title=f'Calibration Curve',
                       save_path=PLOTS_DIR / f'calibration_curve.png')

print("\nAll plots generated successfully!")

## 7. Optuna Study Exploration


In [ ]:


experiments_root = Path("../experiments")
# Find directories matching timestamp pattern
exp_folders = sorted(
    [p for p in experiments_root.iterdir() if p.is_dir() and re.match(r"\d{8}_\d{6}", p.name)],
    key=lambda p: p.name,
    reverse=True
)
if not exp_folders:
    raise FileNotFoundError("No experiment folders found in ../experiments")
latest_exp = exp_folders[0]
print(f"Using experiment folder: {latest_exp}")

study_path = latest_exp / "models" / "optuna_study.pkl"
study = joblib.load(study_path)
print(f"Loaded Optuna study with {len(study.trials)} trials.")


### 7.1 Study Overview


In [ ]:

best = study.best_trial
print(f"Best trial number: {best.number}")
print(f"Best value (competition score): {best.value:.5f}")
print("Best hyperparameters:")
for k, v in best.params.items():
    print(f"  {k}: {v}")


### 7.2 Trials DataFrame


In [ ]:

trials_df = study.trials_dataframe()
# Select a subset of columns for readability
cols = ["number", "value", "params_model_type", "params_learning_rate",
        "params_n_estimators", "params_max_depth", "params_subsample",
        "params_colsample_bytree", "state"]
# Keep only those that exist
existing_cols = [c for c in cols if c in trials_df.columns]
print(trials_df[existing_cols].head(10))
print(f"Total trials: {len(trials_df)}")


### 7.3 Hyperparameter Importance


In [ ]:

try:
    impt = optuna.importance.get_param_importances(study)
    if impt:
        # Sort
        sorted_impt = sorted(impt.items(), key=lambda x: x[1], reverse=True)
        params, importances = zip(*sorted_impt)
        plt.figure(figsize=(8,6))
        plt.barh(params, importances)
        plt.xlabel("Importance")
        plt.title("Hyperparameter Importance")
        plt.gca().invert_yaxis()  # highest on top
        plt.tight_layout()
        plt.show()
    else:
        print("Could not compute parameter importance (no trials with sufficient info).")
except Exception as e:
    print(f"Error plotting parameter importance: {e}")


### 7.4 Optimization History


In [ ]:

try:
    fig = vis.plot_optimization_history(study)
    fig.show()
except Exception as e:
    print(f"Error plotting optimization history: {e}")


### 7.5 Parallel Coordinate Plot


In [ ]:

try:
    fig = vis.plot_parallel_coordinate(study)
    fig.show()
except Exception as e:
    print(f"Error plotting parallel coordinate: {e}")


### 7.6 Slice Plot


In [ ]:

try:
    fig = vis.plot_slice(study)
    fig.show()
except Exception as e:
    print(f"Error plotting slice: {e}")
